In [32]:
import pandas as pd
df = pd.read_csv('Datasets/all_kindle_review.csv')
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [33]:
df=df[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [34]:
df.shape

(12000, 2)

#### Feature Engineering: Data Cleaning and Preprocessing

In [35]:
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [36]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [37]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [38]:
# Let's consider any review which has less then 3 ratings as -ve sentiment else +ve sentiment
df['rating']=df['rating'].apply(lambda x:0 if x<3 else 1)

In [39]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [40]:
df['rating'].value_counts()
# It's not completely an imbalanced dataset.

rating
1    8000
0    4000
Name: count, dtype: int64

In [41]:
# 1. Lower all the cases
df['reviewText']=df['reviewText'].str.lower()

In [42]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [43]:
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup

In [44]:
# 2. Remove Special Characters
df['reviewText']=df['reviewText'].apply(lambda x: re.sub('[^a-zA-Z0-9]+',' ',x))
# 3. Remove all the stopwords
stopwords = set(stopwords.words('english'))
df['reviewText']=df['reviewText'].apply(lambda x: ' '.join([i for i in x.split() if i not in stopwords]))
# 4. Remove url
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
# 5. Remove html contents
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## 6.Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [45]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four books expecting 34 con...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [46]:
# Lemmatizer
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [47]:
def applyLemmatization(x):
    return " ".join([lemmatizer.lemmatize(word) for word in x.split()])

In [48]:
df['reviewText'] = df['reviewText'].apply(lambda x: applyLemmatization(x))

In [49]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four book expecting 34 conc...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


#### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['reviewText'],df['rating'],test_size=0.2)

## Text Embedding

### Bag of Words Vectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

### TF-IDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

### Word2Vec 

In [78]:
from gensim.models import Word2Vec
corpus = [text.split() for text in df['reviewText']] # It should be cleaned corpus
wv_model = Word2Vec(
    sentences=corpus,
    vector_size=300, # Dimensions
    window=5,
    min_count=2, # Ignore words that appear fewer than 2 times.
    workers=4 # Number of CPU cores used during training.
)

In [79]:
import numpy as np
def avg_word2vec(doc):
    words = doc.split()  # if X_train contains strings

    vectors = [
        wv_model.wv[word]
        for word in words
        if word in wv_model.wv.key_to_index
    ]

    if len(vectors) == 0:
        return np.zeros(wv_model.vector_size)

    return np.mean(vectors, axis=0)

In [80]:
X_train_w2v = np.array([avg_word2vec(doc) for doc in X_train])
X_test_w2v = np.array([avg_word2vec(doc) for doc in X_test])

print(X_train_w2v.shape)
print(X_test_w2v.shape)

(9600, 300)
(2400, 300)


In [81]:
print(X_train_bow.shape)
print(X_test_bow.shape)
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(9600, 24627)
(2400, 24627)
(9600, 24627)
(2400, 24627)


## Model Training

In [82]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow = GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf,y_train)
nb_model_w2v = GaussianNB().fit(X_train_w2v,y_train)

In [83]:
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score
y_pred_bow = nb_model_bow.predict(X_test_bow)
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)
y_pred_w2v = nb_model_w2v.predict(X_test_w2v)

In [84]:
print("BOW Accuracy: ",accuracy_score(y_test,y_pred_bow))
print("TF-IDF Accuracy: ",accuracy_score(y_test,y_pred_tfidf))
print("W2V Accuracy: ",accuracy_score(y_test,y_pred_w2v))

BOW Accuracy:  0.57625
TF-IDF Accuracy:  0.58
W2V Accuracy:  0.6875


In [85]:
print("BOW Confusion Matrix: \n",confusion_matrix(y_test,y_pred_bow),"\n")
print("TF-IDF Confusion Matrix: \n",confusion_matrix(y_test,y_pred_tfidf),"\n")
print("W2V Confusion Matrix: \n",confusion_matrix(y_test,y_pred_w2v),"\n")

BOW Confusion Matrix: 
 [[532 265]
 [752 851]] 

TF-IDF Confusion Matrix: 
 [[517 280]
 [728 875]] 

W2V Confusion Matrix: 
 [[ 612  185]
 [ 565 1038]] 



In [86]:
print("BOW Classification Report: \n",classification_report(y_test,y_pred_bow),"\n")
print("TF-IDF Classification Report: \n",classification_report(y_test,y_pred_tfidf),"\n")
print("W2V Classification Report: \n",classification_report(y_test,y_pred_w2v),"\n")

BOW Classification Report: 
               precision    recall  f1-score   support

           0       0.41      0.67      0.51       797
           1       0.76      0.53      0.63      1603

    accuracy                           0.58      2400
   macro avg       0.59      0.60      0.57      2400
weighted avg       0.65      0.58      0.59      2400
 

TF-IDF Classification Report: 
               precision    recall  f1-score   support

           0       0.42      0.65      0.51       797
           1       0.76      0.55      0.63      1603

    accuracy                           0.58      2400
   macro avg       0.59      0.60      0.57      2400
weighted avg       0.64      0.58      0.59      2400
 

W2V Classification Report: 
               precision    recall  f1-score   support

           0       0.52      0.77      0.62       797
           1       0.85      0.65      0.73      1603

    accuracy                           0.69      2400
   macro avg       0.68      0.71 